# ConEdison network forecast

| | |
|---|---|
| **Source** | Consolidated Edison (ConEdison) |
| **URL** | [coned.com](https://www.coned.com/-/media/files/coned/documents/business-partners/local-generation-facilities/systemdata/forecasted-network-data.xlsx) |
| **Granularity** | One row per hour |

The 3-year, 8,760-hour network forecast is developed using network peak demand and energy send-out projections, combined with actual hourly load data from the previous year. This approach captures the embedded impacts of Distributed Energy Resources (DERs) within the service area and informs the load shape modeling for individual networks. Notably, the forecast does not account for the full impact of DER projects that are under construction or pending in the interconnection queue.
 
8,760-hour network forecasts 1) are subject to typical forecasting uncertainties, including—but not limited to—economic conditions, customer behavior, and assumptions regarding future DER adoption; 2) are intended for informational purposes only; 3) are not intended to predict actual hourly loads; and 4) should not be used as a substitute for real-time operational data. 

The forecast is updated every two years, in alignment with the Distributed System Implementation Plan (DSIP) filing schedule.


## Data Acquisition

In [17]:
from datetime import date
from pathlib import Path
from openpyxl import load_workbook
import pandas as pd
import json
import requests

In [18]:
ROOT = Path("..").resolve()
URL = "https://www.coned.com/-/media/files/coned/documents/business-partners/local-generation-facilities/systemdata/forecasted-network-data.xlsx"
PARAMS = None
FILE_PATH = ROOT / "data/raw/coned-forecasted-network-data.xlsx"
META_PATH = ROOT / "data/raw/coned-forecasted-network-data.xlsx.meta.json"

if FILE_PATH.exists():
    print(f"Already fetched → {FILE_PATH}, skipping.")
else:
    FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
    print(f"Fetching dataset → {FILE_PATH}")
    resp = requests.get(URL, params=PARAMS, timeout=30)
    resp.raise_for_status()
    FILE_PATH.write_bytes(resp.content)
    META_PATH.write_text(
        json.dumps(
            {
                "fetched_date": str(date.today()),
                "source_url": URL,
                "source_params": PARAMS,
                "content_bytes": len(resp.content),
            },
            indent=2,
        )
    )
    print(f"Saved {FILE_PATH.name} ({META_PATH.read_text()})")


Already fetched → /home/henrique/source/repos/agentwin-dataset/data/raw/coned-forecasted-network-data.xlsx, skipping.


In [ ]:
def read_forecast(file_path, sheet_name, nrows=8760):
    def _get_forecast_columns(file_path, sheet_name):
        workbook = load_workbook(
            file_path,
            read_only=True,
            data_only=True,
        )

        try:
            worksheet = (
                workbook.worksheets[sheet_name]
                if isinstance(sheet_name, int)
                else workbook[sheet_name]
            )

            # Excel A19
            first_column = next(
                worksheet.iter_rows(
                    min_row=19,
                    max_row=19,
                    min_col=1,
                    max_col=1,
                    values_only=True,
                )
            )[0]

            # Excel B16:CR16
            remaining_columns = next(
                worksheet.iter_rows(
                    min_row=16,
                    max_row=16,
                    min_col=2,
                    max_col=96,  # CR
                    values_only=True,
                )
            )
        finally:
            workbook.close()

        columns = [first_column, *remaining_columns]

        if any(column is None for column in columns):
            raise ValueError("One or more forecast headers are empty")

        return [str(column).strip() for column in columns]

    columns = _get_forecast_columns(file_path=file_path, sheet_name=sheet_name)

    forecast = pd.read_excel(
        file_path,
        engine="openpyxl",
        usecols="A:CR",
        skiprows=19,
        header=None,
        names=columns,
        sheet_name=sheet_name,
        nrows=nrows,
        dtype={column: "float64" for column in columns[1:]},
    )

    forecast[columns[0]] = pd.to_datetime(
        forecast[columns[0]],
        format="%m/%d/%Y %H:%M",
        errors="raise",
    )

    return forecast


forecast_2025_2027_df: pd.DataFrame = pd.concat(
    [
        read_forecast(FILE_PATH, sheet_name=0),
        read_forecast(FILE_PATH, sheet_name=1),
        read_forecast(FILE_PATH, sheet_name=2),
    ],
)

# Write to file
forecast_2025_2027_df.to_csv(
    ROOT / "data/processed/coned_forecast_network.csv",
)

forecast_2025_2027_df

,Battery Park City,Bay Ridge,Beekman,Borden,Borough Hall,Bowling Green,Brighton Beach,Buchanan,Canal,Cedar Street,...,Co-op City/ Parkchester 138KV,Greenwood,Parkchester 2,Plymouth Street,Sunnyside Amtrak,Mount Vernon MTA,Van Nest Amtrak,Glendale,Corona 1,Lenox Hill Radial
Hour Beginning,,,,,,,,,,,,,,,,,,,,,
2025-01-01 00:00:00,24.648544,101.332049,35.508510,55.888491,132.611315,32.411370,43.821349,44.837310,44.654992,40.074396,...,4.774617,8.236658,17.825541,21.532154,8.233479,1.757348,0.095236,12.593489,11.414868,0.655061
2025-01-01 01:00:00,24.039520,98.717709,34.867356,55.409268,129.884813,32.215098,42.349215,43.620961,44.145139,38.736645,...,4.391155,7.612386,17.377386,20.962285,7.297814,3.247266,0.110367,12.063851,10.958411,0.670059
2025-01-01 02:00:00,23.505046,96.381396,34.268114,54.343263,126.085828,31.324095,40.946010,42.396209,43.330580,37.323583,...,4.171461,7.060260,16.952601,20.466501,7.115583,2.700974,0.106587,11.564821,10.710858,0.678088
2025-01-01 03:00:00,23.470191,94.761483,34.057757,53.298351,124.421225,30.988108,39.847808,41.551510,42.929163,36.552348,...,4.171461,6.675415,16.555962,20.319133,7.082283,2.236089,0.110367,11.156328,10.250045,0.676087
2025-01-01 04:00:00,23.665740,94.318959,34.107371,53.075610,124.056771,30.940733,39.644101,41.445736,42.864432,36.154452,...,4.171461,6.451097,16.382273,20.232678,7.129457,0.865337,0.106587,10.917552,10.058054,0.713504
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2027-12-31 19:00:00,30.185789,124.156262,42.682967,64.886043,168.404286,42.277995,54.497744,51.975344,48.771220,48.564142,...,14.270843,10.311541,22.138431,23.684698,7.285077,5.416647,0.674252,14.022282,11.894865,1.326073
2027-12-31 20:00:00,29.026836,122.021787,40.176210,63.956581,162.034593,40.057681,54.604307,50.125771,47.561680,47.509310,...,14.186861,10.362122,21.413432,23.067956,9.489412,3.193708,0.682547,13.745648,12.084652,1.328248
2027-12-31 21:00:00,28.148702,119.207182,38.663952,62.781905,156.269845,38.900464,52.167823,47.669990,46.795950,46.054362,...,13.857307,10.186807,21.145959,23.104471,11.317127,3.179920,0.034294,13.518493,11.199576,1.354362
